![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Taller: Construcción e implementación de modelos Bagging, Random Forest y XGBoost

En este taller podrán poner en práctica sus conocimientos sobre la construcción e implementación de modelos de Bagging, Random Forest y XGBoost. El taller está constituido por 8 puntos, en los cuales deberan seguir las intrucciones de cada numeral para su desarrollo.

## Datos predicción precio de automóviles

En este taller se usará el conjunto de datos de Car Listings de Kaggle donde cada observación representa el precio de un automóvil teniendo en cuenta distintas variables como año, marca, modelo, entre otras. El objetivo es predecir el precio del automóvil. Para más detalles puede visitar el siguiente enlace: [datos](https://www.kaggle.com/jpayne/852k-used-car-listings).

In [8]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
# Importación de librerías
%matplotlib inline
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
# Lectura de la información de archivo .csv
data = pd.read_csv('https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/datasets/dataTrain_carListings.zip')

# Preprocesamiento de datos para el taller
data = data.loc[data['Model'].str.contains('Camry')].drop(['Make', 'State'], axis=1)
data = data.join(pd.get_dummies(data['Model'], prefix='M'))
data = data.drop(['Model'], axis=1)

# Visualización dataset
data.head()

,Price,Year,Mileage,M_Camry,M_Camry4dr,M_CamryBase,M_CamryL,M_CamryLE,M_CamrySE,M_CamryXLE
7,21995,2014,6480,False,False,False,True,False,False,False
11,13995,2014,39972,False,False,False,False,True,False,False
167,17941,2016,18989,False,False,False,False,False,True,False
225,12493,2014,51330,False,False,False,True,False,False,False
270,7994,2007,116065,False,True,False,False,False,False,False


In [10]:
# Separación de variables predictoras (X) y variable de interés (y)
y = data['Price']
X = data.drop(['Price'], axis=1)

In [11]:
# Separación de datos en set de entrenamiento y test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

### Punto 1 - Árbol de decisión manual

En la celda 1 creen un árbol de decisión **manualmente**  que considere los set de entrenamiento y test definidos anteriormente y presenten el RMSE y MAE del modelo en el set de test.

In [12]:
# Celda 1
# Definición de parámetros y criterios de parada
max_depth = 5 # profundidad maxima del arbol
num_pct = 10 # número de percentiles a evaluar para generar los splits
max_features = None 
min_gain=0.001

In [13]:
# Impresión variable a usar (Kilometraje)
j = 1
print(X.columns[j])

Mileage


In [14]:
# División de la variable Mileage (kilometraje) en num_ctp puntos (parámetro definido anteriormente) para obtener posibles puntos de corte
splits = np.percentile(X.iloc[:, j], np.arange(0, 100, 100.0 / num_pct).tolist())
splits = np.unique(splits)
splits

array([5.000000e+00, 1.576440e+04, 2.345860e+04, 2.993760e+04,
       3.570080e+04, 4.168000e+04, 4.824700e+04, 6.171600e+04,
       8.153420e+04, 1.067578e+05])

In [15]:
# División de las observaciones usando el punto de corte en la posición 5 de la lista de splits
k=5
filter_l = X.iloc[:, j] < splits[k]

y_l = y.loc[filter_l]
y_r = y.loc[~filter_l]

In [16]:
# Definición de la función gini_imputiry para calular la ganancia de una variable predictora j dado el punto de corte k
def mse(y):
    if len(y) == 0:
        return 0
    return ((y - y.mean())**2).mean()


def mse_gain(x, y, split):
    
    filter_l = x < split
    filter_r = ~filter_l
    
    y_l = y[filter_l]
    y_r = y[filter_r]
    
    n = len(y)
    n_l = len(y_l)
    n_r = len(y_r)
    
    if n_l == 0 or n_r == 0:
        return 0
    
    mse_parent = mse(y)
    
    mse_children = (n_l/n) * mse(y_l) + (n_r/n) * mse(y_r)
    
    gain = mse_parent - mse_children
    
    return gain
# Definición de la función best_split para encontrar la mejor variable predictora j y el mejor punto de Corte

def best_split(X, y, num_pct=10):
    
    features = range(X.shape[1])
    
    best_split = [0, 0, 0]  # j, split, gain
    
    features = X.select_dtypes(include=[np.number]).columns

    for j in features:
    
     splits = np.percentile(X[j], np.arange(0, 100, 100.0 / (num_pct+1)))
     splits = np.unique(splits)[1:]
    
     for split in splits:
        gain = mse_gain(X[j], y, split) 
        
        if gain > best_split[2]:
            best_split = [j, split, gain]

    return best_split


In [17]:
# Obtención de la variable 'j', su punto de corte 'split' y su ganancia 'gain'
j, split, gain = best_split(X, y, 5)
j, split, gain

('Year', np.float64(2014.0), np.float64(8674619.810995983))

In [18]:
# División de las observaciones usando la mejor variable 'j' y su punto de corte 'split'
filter_l = X[j] < split

y_l = y.loc[filter_l]
y_r = y.loc[~filter_l]

In [19]:
# Definición de la función tree_grow para hacer un crecimiento recursivo del árbol
def tree_grow(X, y, level=0, min_gain=0.001, max_depth=5, num_pct=10):
    
    # Si solo es una observación
    if X.shape[0] == 1:
        tree = dict(y_pred=y.iloc[0], level=level, split=-1, n_samples=1, gain=0)
        return tree
    
    # Calcular la mejor división
    j, split, gain = best_split(X, y, num_pct)
    
    # Guardar el árbol y estimar la predicción
    y_pred = y.mean()  # predicción es el promedio
    
    tree = dict(y_pred=y_pred, level=level, split=-1, n_samples=X.shape[0], gain=gain)
    # Revisar el criterio de parada 
    if gain < min_gain:
        return tree
    if max_depth is not None:
        if level >= max_depth:
            return tree   
    
    # Continuar creando la partición
    filter_l = X[j] < split
    X_l, y_l = X.loc[filter_l], y.loc[filter_l]
    X_r, y_r = X.loc[~filter_l], y.loc[~filter_l]
    tree['split'] = [j, split]

    # Siguiente iteración para cada partición
    
    tree['sl'] = tree_grow(X_l, y_l, level + 1, min_gain=min_gain, max_depth=max_depth, num_pct=num_pct)
    tree['sr'] = tree_grow(X_r, y_r, level + 1, min_gain=min_gain, max_depth=max_depth, num_pct=num_pct)
    
    return tree

In [20]:
# Aplicación de la función tree_grow
tree = tree_grow(X, y, max_depth=5, min_gain=0.01)
print(tree['split'], tree['gain'])

['Year', np.float64(2014.0)] 8674619.810995983


In [21]:
# Definición de la función tree_predict para hacer predicciones según las variables 'X' y el árbol 'tree'

def tree_predict(X, tree):

    # Nodo hoja
    if tree['split'] == -1:
        return np.full(X.shape[0], tree['y_pred'])
    
    # Nodo interno
    j, split = tree['split']
    filter_l = X[j] < split

    X_l = X.loc[filter_l]
    X_r = X.loc[~filter_l]

    # Inicializar predicciones
    y_pred = np.zeros(X.shape[0])

    # Recursión
    if len(X_l) > 0:
        y_pred[filter_l] = tree_predict(X_l, tree['sl'])
    
    if len(X_r) > 0:
        y_pred[~filter_l] = tree_predict(X_r, tree['sr'])

    return y_pred

In [22]:
y_mean = y_train.mean()
y_pred_baseline = np.full(len(y_test), y_mean)

rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
mae_baseline = mean_absolute_error(y_test, y_pred_baseline)

#  Entrenamiento 
tree = tree_grow(X_train, y_train, max_depth=5, min_gain=0.01)

# Predicción 
y_pred_train = tree_predict(X_train, tree)
y_pred_test  = tree_predict(X_test, tree)

# Métricas
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test, y_pred_test))

mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test  = mean_absolute_error(y_test, y_pred_test)

# Resultados
print("===== BASELINE =====")
print(f"RMSE baseline: {rmse_baseline:.4f}")
print(f"MAE  baseline: {mae_baseline:.4f}")

print("\n===== ÁRBOL DE DECISIÓN =====")
print(f"RMSE train: {rmse_train:.4f}")
print(f"RMSE test : {rmse_test:.4f}")
print(f"MAE  train: {mae_train:.4f}")
print(f"MAE  test : {mae_test:.4f}")


print("\n===== DIAGNÓSTICO =====")

if rmse_test < rmse_baseline:
    print(" El modelo mejora sobre el baseline")
else:
    print(" El modelo NO mejora el baseline")

if rmse_train < rmse_test:
    print("Posible overfitting (train mejor que test)")
else:
    print(" No hay señales claras de overfitting")

===== BASELINE =====
RMSE baseline: 3912.1081
MAE  baseline: 3111.6222

===== ÁRBOL DE DECISIÓN =====
RMSE train: 1673.4891
RMSE test : 1742.1476
MAE  train: 1249.9360
MAE  test : 1293.2123

===== DIAGNÓSTICO =====
 El modelo mejora sobre el baseline
Posible overfitting (train mejor que test)


### Procedimiento 
Se implemento un árbol de regresión de firma manual para predecir el precio de automóviles apartir de variables como año , kilometraje y versión del modelo. 
### Resultado
Al comparar el desempeño entre entrenamiento y prueba, se observa que el error en entrenamiento (RMSE de 1673 y MAE de 1249) es ligeramente menor que en el conjunto de prueba. Aunque esto podría sugerir un posible sobreajuste, la diferencia entre ambos conjuntos es pequeña, por lo que no se considera que exista un problema real de overfitting. En cambio, el modelo muestra un comportamiento estable y una buena capacidad de generalización.

### Punto 2 - Bagging manual

En la celda 2 creen un modelo bagging **manualmente** con 10 árboles de regresión y comenten sobre el desempeño del modelo.

In [23]:
# Celda 2


### Punto 3 - Bagging con librería

En la celda 3, con la librería sklearn, entrenen un modelo bagging con 10 árboles de regresión y el parámetro `max_features` del árbol de decisión igual a `log(n_features)` y comenten sobre el desempeño del modelo.

In [24]:
# Celda 3


### Punto 4 - Random forest con librería

En la celda 4, usando la librería sklearn entrenen un modelo de Randon Forest para regresión  y comenten sobre el desempeño del modelo.

In [25]:
# Celda 4


### Punto 5 - Calibración de parámetros Random forest

En la celda 5, calibren los parámetros max_depth, max_features y n_estimators del modelo de Randon Forest para regresión, comenten sobre el desempeño del modelo y describan cómo cada parámetro afecta el desempeño del modelo.

In [26]:
# Celda 5


### Punto 6 - XGBoost con librería

En la celda 6 implementen un modelo XGBoost de regresión con la librería sklearn y comenten sobre el desempeño del modelo.

In [27]:
# Celda 6


### Punto 7 - Calibración de parámetros XGBoost

En la celda 7 calibren los parámetros learning rate, gamma y colsample_bytree del modelo XGBoost para regresión, comenten sobre el desempeño del modelo y describan cómo cada parámetro afecta el desempeño del modelo.

In [28]:
# Celda 7


### Punto 8 - Comparación y análisis de resultados
En la celda 8 comparen los resultados obtenidos de los diferentes modelos (random forest y XGBoost) y comenten las ventajas del mejor modelo y las desventajas del modelo con el menor desempeño.

In [29]:
# Celda 8
